# k-means on weighted points

A bakery chain wants four delivery hubs. It knows how many orders each of 300 city blocks
placed last month, and a hub should sit close to the orders, not to the blocks: a block with
40 orders counts 40 times, and a block with none should not pull a hub at all. You will
write the weighted update step, Lloyd's algorithm on weighted points, and k-means++ seeding
that respects the weights.

The weighted objective is

$$
J = \sum_{i} w_i \, \lVert \mathbf{x}_i - \boldsymbol{\mu}_{c(i)} \rVert^2 ,
$$

where $w_i$ is the number of orders from block $i$ and $c(i)$ is the hub it is assigned to.
It is exactly the plain k-means objective on a list in which block $i$ appears $w_i$ times.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# 300 blocks on a 20 by 15 grid (coordinates in km) and the orders each placed last month.
gx, gy = np.meshgrid(np.arange(20.0), np.arange(15.0))
X = np.column_stack([gx.ravel(), gy.ravel()])
towns = np.array([[3.0, 3.0], [15.0, 4.0], [5.0, 11.0], [14.0, 12.0]])
rate = sum(30 * np.exp(-((X - t) ** 2).sum(axis=1) / 4.0) for t in towns)
w = np.random.default_rng(8).poisson(rate).astype(int)
print(f"{len(X)} blocks, {w.sum()} orders, {np.sum(w == 0)} blocks with no orders")

## Part 1 (coding task): The weighted update step

Write `weighted_update(X, w, labels, C)`. `X` holds the blocks, `w` their order counts,
`labels[i]` the hub block `i` is assigned to, and `C` the current hub positions, one row per
hub. Return a new array of hub positions in which each hub moves to the position that
minimizes its share of $J$ with the assignment held fixed. A hub whose blocks placed no
orders at all, or that has no blocks, keeps its current position. Do not change `C` itself.

In [ ]:
def weighted_update(X, w, labels, C):
    """New hub positions for a fixed assignment."""
    # TODO
    raise NotImplementedError

In [ ]:
Xs = np.array([[0, 0], [4, 0], [0, 4], [10, 10], [12, 10], [30, 30]], dtype=float)
ws = np.array([1, 1, 2, 3, 1, 0])
Cs = np.array([[1, 1], [9, 9], [50, 50], [29, 29]], dtype=float)
before = Cs.copy()
new = weighted_update(Xs, ws, np.array([0, 0, 0, 1, 1, 3]), Cs)
assert np.allclose(Cs, before), "weighted_update changed C in place: return a new array"
assert np.allclose(new[0], [1, 2]), f"hub 0 should move to (1, 2), got {new[0]}: is it the plain mean of its blocks instead of the order-weighted mean?"
assert np.allclose(new[1], [10.5, 10]), f"hub 1 should move to (10.5, 10), got {new[1]}"
assert np.allclose(new[2], [50, 50]), f"hub 2 has no blocks and should stay at (50, 50), got {new[2]}"
assert np.allclose(new[3], [29, 29]), f"hub 3's only block placed no orders, so it should stay at (29, 29), got {new[3]}"

lab = np.random.default_rng(1).integers(0, 4, len(X))
C4 = X[[22, 37, 230, 255]]
Xe, labe = np.repeat(X, w, axis=0), np.repeat(lab, w)
expected = np.stack([Xe[labe == j].mean(axis=0) for j in range(4)])
assert np.allclose(weighted_update(X, w, lab, C4), expected), "on the city data the hubs should match the plain means of the list with each block repeated w times"
print("Part 1 passed")

## Part 2 (coding task): Lloyd's algorithm on weighted points

Write `weighted_kmeans(X, w, C0, max_rounds=100)`. Start from the hub positions `C0` and
repeat: assign every block to its nearest hub (squared distance, ties to the lower index as
`argmin` does), then move the hubs with `weighted_update`. Stop when an assign step gives
the same assignment as the step before, or after `max_rounds` rounds. Return
`(labels, C, J)`: the final assignment, the final hub positions, and the weighted inertia
$J$ of that assignment with those hubs.

In [ ]:
def weighted_kmeans(X, w, C0, max_rounds=100):
    """Lloyd's algorithm with order weights. Returns (labels, C, J)."""
    # TODO
    raise NotImplementedError

In [ ]:
def plain_lloyd(P, C, rounds=100):
    # Reference: unweighted Lloyd, an empty cluster keeps its center.
    C = np.array(C, dtype=float)
    for _ in range(rounds):
        lab = ((P[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        C = np.stack([P[lab == j].mean(axis=0) if np.any(lab == j) else C[j] for j in range(len(C))])
    lab = ((P[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
    return C, float(((P - C[lab]) ** 2).sum())

Xe = np.repeat(X, w, axis=0)
for start in ([0, 19, 280, 299], [140, 145, 150, 155], [22, 37, 230, 255]):
    labels, C, J = weighted_kmeans(X, w, X[start])
    C_ref, J_ref = plain_lloyd(Xe, X[start])
    assert np.allclose(C, C_ref, atol=1e-6), f"from blocks {start} the hubs should end at\n{np.round(C_ref, 3)}\nbut ended at\n{np.round(C, 3)}"
    assert np.isclose(J, J_ref, rtol=1e-9), f"from blocks {start} the weighted inertia should be {J_ref:.3f}, got {J:.3f}: weight each squared distance by the block's orders"
    assert np.array_equal(labels, ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)), "return the assignment that goes with the final hubs"

print("Part 2 passed")

## Part 3 (coding task): k-means++ seeding with weights

Write `weighted_kmeanspp(X, w, k, rng)`, which returns an array of `k` row indices of `X`
to use as starting hubs. Run on the repeated list, k-means++ would draw its first seed
uniformly from all the orders, then draw every further seed with a chance proportional to
$D(\mathbf{x})^2$, where $D$ is the distance from an order to its nearest seed so far. Do the
same on the weighted blocks: choose the first block with probability proportional to
$w_i$ and each later block with probability proportional to $w_i D(\mathbf{x}_i)^2$. Draw
with `rng.choice(len(X), p=...)`.

The check draws a few thousand seedings on a small example, and then picks the hubs from ten
seeded runs.

In [ ]:
def weighted_kmeanspp(X, w, k, rng):
    """Indices of k starting blocks, drawn as k-means++ draws from the repeated list."""
    # TODO
    raise NotImplementedError

In [ ]:
for s in range(100):
    idx = np.asarray(weighted_kmeanspp(X, w, 4, np.random.default_rng(s)))
    assert idx.shape == (4,), f"expected 4 indices, got shape {idx.shape}"
    assert len(set(idx.tolist())) == 4, f"seeding {s} chose a block twice: {idx}. Measure D to the nearest seed chosen so far"
    assert np.all(w[idx] > 0), f"seeding {s} chose a block with no orders: {idx}. A block with w = 0 should never be drawn"

# A small case with exact probabilities: blocks at 0, 1, 4 and 9 km with 3, 1, 2 and 2 orders.
Xq = np.array([[0.0, 0.0], [1.0, 0.0], [4.0, 0.0], [9.0, 0.0]])
wq = np.array([3, 1, 2, 2])
draws = np.array([weighted_kmeanspp(Xq, wq, 2, np.random.default_rng(1000 + s)) for s in range(4000)])
first = np.bincount(draws[:, 0], minlength=4) / len(draws)
assert np.allclose(first, wq / wq.sum(), atol=0.03), f"first seed frequencies {np.round(first, 3)}, expected {wq / wq.sum()}: draw it with probability proportional to w"
after0 = draws[draws[:, 0] == 0, 1]
p_exact = wq * (Xq[:, 0] - 0.0) ** 2 / (wq * (Xq[:, 0] - 0.0) ** 2).sum()
freq = np.bincount(after0, minlength=4) / len(after0)
assert np.allclose(freq, p_exact, atol=0.04), f"after a first seed at 0 km the second seed frequencies are {np.round(freq, 3)}, expected {np.round(p_exact, 3)}: weight by w times the squared distance"

best = min((weighted_kmeans(X, w, X[weighted_kmeanspp(X, w, 4, np.random.default_rng(s))]) for s in range(10)), key=lambda r: r[2])
labels, hubs, J = best
_, _, J_towns = weighted_kmeans(X, w, towns)
print("hubs (km):", np.round(hubs[np.argsort(hubs[:, 0] + 100 * hubs[:, 1])], 2).tolist())
print(f"weighted inertia: best of ten seeded runs {J:.1f}, started from the four town centers {J_towns:.1f}")
assert J <= J_towns * 1.001, "the best of ten seeded runs should do as well as starting at the town centers"
print("Part 3 passed")

## Solution

### Part 1: The weighted update step

In [ ]:
def weighted_update(X, w, labels, C):
    """New hub positions for a fixed assignment."""
    new = np.array(C, dtype=float)
    for j in range(len(new)):
        mask = labels == j
        total = w[mask].sum()
        if total > 0:
            new[j] = (w[mask, None] * X[mask]).sum(axis=0) / total
    return new

**Why it works.** With the assignment fixed, hub $j$'s share of $J$ is
$\sum_{i \in C_j} w_i \lVert \mathbf{x}_i - \boldsymbol{\mu} \rVert^2$. Its gradient,
$-2 \sum_{i \in C_j} w_i (\mathbf{x}_i - \boldsymbol{\mu})$, is zero at the weighted mean, the same
derivation as for the plain mean with each point counted $w_i$ times. A cluster with total
weight zero contributes nothing to $J$ wherever its hub is, so leaving the hub alone is safe
and avoids dividing by zero.

### Part 2: Lloyd's algorithm on weighted points

In [ ]:
def weighted_kmeans(X, w, C0, max_rounds=100):
    """Lloyd's algorithm with order weights. Returns (labels, C, J)."""
    C = np.array(C0, dtype=float)
    labels = None
    for _ in range(max_rounds):
        new_labels = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2).argmin(axis=1)
        if labels is not None and np.array_equal(new_labels, labels):
            break
        labels = new_labels
        C = weighted_update(X, w, labels, C)
    J = float((w * ((X - C[labels]) ** 2).sum(axis=1)).sum())
    return labels, C, J

**Why it works.** Each pass of the loop does the two steps of Lloyd's algorithm, and the
weighted update minimizes the weighted objective just as the plain mean minimizes the plain
one, so $J$ never increases and the loop stops. Because a weighted point behaves exactly
like $w_i$ copies of one point, the run matches plain Lloyd's algorithm on the repeated list
step for step, which is what the check compares against, while touching 300 rows instead of
about fifteen hundred.

### Part 3: k-means++ seeding with weights

In [ ]:
def weighted_kmeanspp(X, w, k, rng):
    """Indices of k starting blocks, drawn as k-means++ draws from the repeated list."""
    w = np.asarray(w, dtype=float)
    seeds = [rng.choice(len(X), p=w / w.sum())]
    d2 = ((X - X[seeds[0]]) ** 2).sum(axis=1)
    for _ in range(k - 1):
        p = w * d2
        seeds.append(rng.choice(len(X), p=p / p.sum()))
        d2 = np.minimum(d2, ((X - X[seeds[-1]]) ** 2).sum(axis=1))
    return np.array(seeds)

**Why it works.** On the repeated list, block $i$ is drawn whenever any of its $w_i$ copies is, so its
chance is $w_i$ times the chance of one copy: proportional to $w_i$ for the first seed and to
$w_i D(\mathbf{x}_i)^2$ after that. Keeping `d2` as the running minimum over the seeds so far gives
the distance to the nearest seed in one `np.minimum` per seed. A block with no orders gets
probability zero in every draw, and a block already chosen has $D = 0$, so it cannot be drawn
again. Ignoring the weights would often start a hub on an empty outlying block, since those
are the blocks farthest from everything.